# Pooling, Robustness & Overfitting in CNNs

**Theme:**

*“Convolution finds patterns. Pooling keeps only the strongest ones.”*

## Imports

In [27]:
import torch 
import torch.nn as nn 
from torchvision import datasets, transforms
from torch.utils.data import DataLoader


torch.__version__

'2.8.0+cu129'

## Part 1 - What MaxPool actually does

- We used:

```python
nn.MaxPool2d(2)
```

- This converts:

```python
28×28 → 14×14
14×14 → 7×7
```

- But WHY?

### Conceptual Understanding

In [2]:
# Let's take
x = torch.tensor([[1,5],
                  [2,3]]) # 2x2 tensor

output = torch.max(x) # maxpooling , 2x2 -> 1x1
print(output)

tensor(5)


Answer in own words:

1) What happens inside a 2×2 MaxPool?
2) What information is being kept?
3) What information is being discarded?

### Why pooling helps CNNs

Think of this situation:

- A digit "3" is slightly shifted:

  - Left

  - Right

  - Up

- CNN should still recognize it.

- Pooling helps with:

   - small position changes

📌 Answer:

- How does MaxPool make CNNs more position-robust?

## Part 2 - Removing pooling (EXPERIMENT)

### Modify CNN

In [ ]:
# Data
transform = transforms.ToTensor()

train_data = datasets.MNIST(root='../week2/data', download=True, train=True, transform=transform)
test_data = datasets.MNIST(root='../week2/data', download=True, train=False, transform=transform) 

train_loader = DataLoader(train_data, batch_size=100, shuffle=True) 
test_loader = DataLoader(test_data, batch_size=100, shuffle=False) 

# Network (pooling modified)
class MyCNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1, stride=1),  # 1(black&white)28x28x1 -> 28x28x32 *formula:- (width - filter + 2 x pdding)/stride + 1
            nn.ReLU(),

            nn.Conv2d(32, 64, kernel_size=3, padding=1, stride=1), # 28x28x32 => 28x28x64
            nn.ReLU()
        )
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64*28*28, 512),
            nn.ReLU(),
            nn.Linear(512,64),
            nn.ReLU(),
            nn.Linear(64,10)
        )
    
    def forward(self,x):
        x = self.conv(x)
        output = self.fc(x)
        return output 


### Observe

1) What happened to:

- Training speed?

- Memory usage?

2) Accuracy?

Why pooling helps computationally?

**MOVING THE MODEL TO GPU**

In [20]:
device = None 
if torch.cuda.is_available():
    device = 'cuda:0'
else:
    print("Cuda isn't available")
    device = 'cpu'
print(device)


cuda:0


In [21]:
# model, loss function and optmizer
model = MyCNN().to(device) 
loss_fn = nn.CrossEntropyLoss() 
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3) 

In [22]:
#####################################################   TRAIN   ###############################################################
model.train()

for i in range(10):
    total_loss = 0.0 

    for images, labels in train_loader:
        images = torch.tensor(images, device=device)
        labels = torch.tensor(labels, device=device)

        outputs = model(images) 
        loss = loss_fn(outputs, labels) 

        optimizer.zero_grad()
        loss.backward()
        optimizer.step() 

        total_loss += loss.item()
    print(f"Epoch: {i+1} | Train Loss: {total_loss/len(train_loader):.6f}")


###################################################    EVALUATION    ###################################################  
model.eval() 

total = 0
correct = 0

with torch.no_grad():
    total_loss = 0.0

    for images,labels in test_loader:
        images = torch.tensor(images, device=device)
        labels = torch.tensor(labels, device=device)

        outputs = model(images) 
        loss = loss_fn(outputs,labels) 

        prediction = outputs.argmax(dim=1) 
        correct += (prediction==labels).sum().item() 
        total += len(labels) 

        total_loss += loss.item() 
    
    print(f"Test Loss: {total_loss/len(test_loader):.6f} | Test Accuracy: {correct/total * 100:.2f} %")




Epoch: 1 | Train Loss: 0.187775
Epoch: 2 | Train Loss: 0.043132
Epoch: 3 | Train Loss: 0.024175
Epoch: 4 | Train Loss: 0.015853
Epoch: 5 | Train Loss: 0.010075
Epoch: 6 | Train Loss: 0.009680
Epoch: 7 | Train Loss: 0.009301
Epoch: 8 | Train Loss: 0.005521
Epoch: 9 | Train Loss: 0.004879
Epoch: 10 | Train Loss: 0.006505
Test Loss: 0.041664 | Test Accuracy: 98.98 %


## CNNs CAN overfit too

### Modify in flatten hidden layer

In [11]:
# Network (Add a dropout layer in fc)
class ModfCNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1, stride=1),  # 1(black&white)28x28x1 -> 28x28x32 *formula:- (width - filter + 2 x pdding)/stride + 1
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),                                # 28x28x32 -> 14x14x32

            nn.Conv2d(32, 64, kernel_size=3, padding=1, stride=1), # 14x14x32 => 14x14x64
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)                                   # 14x14 => 7x7
        )
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64*7*7, 512),
            nn.ReLU(),
            nn.Linear(512,64),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(64,10)
        )
    
    def forward(self,x):
        x = self.conv(x)
        output = self.fc(x)
        return output 


### Model Training

In [25]:
model2 = ModfCNN().to(device) 
optimizer2 = torch.optim.Adam(model2.parameters(), lr=1e-3)

In [ ]:
# ignoring warnings
import warnings
warnings.filterwarnings('ignore')

In [26]:
#####################################################   TRAIN   ###############################################################
model2.train()

for i in range(10):
    total_loss = 0.0 

    for images, labels in train_loader:
        gpu_img = torch.tensor(images, device=device)
        gpu_lbl = torch.tensor(labels, device=device)

        outputs = model2(gpu_img) 
        loss = loss_fn(outputs, gpu_lbl) 

        optimizer2.zero_grad()
        loss.backward()
        optimizer2.step() 

        total_loss += loss.item()
    print(f"Epoch: {i+1} | Train Loss: {total_loss/len(train_loader):.6f}")


###################################################    EVALUATION    ###################################################  
model2.eval() 

total = 0
correct = 0

with torch.no_grad():
    total_loss = 0.0

    for images,labels in test_loader:
        images = torch.tensor(images, device=device)
        labels = torch.tensor(labels, device=device)

        outputs = model2(images) 
        loss = loss_fn(outputs,labels) 

        prediction = outputs.argmax(dim=1) 
        correct += (prediction==labels).sum().item() 
        total += len(labels) 

        total_loss += loss.item() 
    
    print(f"Test Loss: {total_loss/len(test_loader):.6f} | Test Accuracy: {correct/total * 100:.2f} %")




Epoch: 1 | Train Loss: 0.328888
Epoch: 2 | Train Loss: 0.091383
Epoch: 3 | Train Loss: 0.059569
Epoch: 4 | Train Loss: 0.045109
Epoch: 5 | Train Loss: 0.035983
Epoch: 6 | Train Loss: 0.029997
Epoch: 7 | Train Loss: 0.023403
Epoch: 8 | Train Loss: 0.022473
Epoch: 9 | Train Loss: 0.015838
Epoch: 10 | Train Loss: 0.014795
Test Loss: 0.039316 | Test Accuracy: 99.21 %


After train 10 iterations.
```python
Epoch: 1 | Train Loss: 0.357495
Epoch: 2 | Train Loss: 0.094169
Epoch: 3 | Train Loss: 0.062190
Epoch: 4 | Train Loss: 0.046313
Epoch: 5 | Train Loss: 0.036595
Epoch: 6 | Train Loss: 0.029247
Epoch: 7 | Train Loss: 0.022619
Epoch: 8 | Train Loss: 0.020020
Epoch: 9 | Train Loss: 0.018091
Epoch: 10 | Train Loss: 0.014726
Test Loss: 0.032699 | Test Accuracy: 99.33 %
```

## Think like a researcher

📌 Answer:

- Why can CNNs still overfit?

- Why does dropout help even in CNNs?

# Interaction

### What MaxPool keeps/discards


MNIST pixels are grayscale (0 = black, 1 = white after ToTensor normalization)

But CNN is not thinking in terms of color.

It is working on feature activations, not raw pixels.

So in practice:

- MaxPool keeps the strongest feature response in that region.

Example:
- If a filter detects an edge, MaxPool keeps:

  - the strongest edge response

  - ignores weaker nearby responses

- So it's more like:

  - “Keep the strongest signal of a pattern.”

---

### MaxPool vs MeanPool (Average Pooling)

#### MaxPool


Keeps:

- Strongest activation

- Most dominant feature

Used when:

- Presence of a feature matters more than exact distribution

Example:

- Edge present? → important

- Exact pixel intensity? → less important

This is why MaxPool is most common.

---

#### MeanPool (Average Pooling)


Takes:

- average of values in region

Used when:

- Overall texture matters

- Not just strongest signal

More common in:

- Older CNNs

- Some classification architectures

- Global average pooling (very popular)

Intuition difference

> MaxPool asks:

“Is this feature present here?”

> MeanPool asks:

“How strong is this region overall?”

---

### Why pooling gives position robustness


If a feature shifts slightly:

Without pooling:

- Activation position changes

- Model may treat it differently

With pooling:

- Small shifts still produce similar max values

- Model becomes less sensitive to exact location

This is called:

- **Translation invariance**

---

### Benefits of pooling

Without pooling observation:

- Training slower ✔

- Memory increased ✔

- Accuracy slightly worse ✔

**Why?**

Without pooling:

- Feature maps remain 28×28

- Huge number of parameters in FC layer:

- 64×28×28 = 50,176 inputs

vs with pooling:

- 64×7×7 = 3,136 inputs

That’s a massive difference.

So pooling:

- Reduces computation

- Reduces overfitting

- Speeds training

---

### CNN overfitting explanation


“Dropout decreases variance and increases bias”


Simpler DL wording:

- Dropout forces the network to not depend too much on specific neurons, so it learns more general patterns.